## Step 1: Import libraries and define S3 paths
### What this step does

This sets up the notebook, imports all required libraries, and defines where your cleaned data lives and where the train, validation, and test files will be saved in S3.

### Why this step matters

Your design document already says the team uses SageMaker Studio with S3 and keeps raw and processed data in the same bucket. So this step keeps the workflow consistent.

In [1]:
import os
import io
import boto3
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

bucket = "retailpulse-team3-ads508"

cleaned_key = "processed-data/cleaned_retail_data.csv"

train_key = "processed-data/train/train.csv"
val_key   = "processed-data/validation/validation.csv"
test_key  = "processed-data/test/test.csv"

s3 = boto3.client("s3")

## Step 2: Load the cleaned dataset from S3
### What this step does

This reads the cleaned dataset your earlier notebook already created.

### Why this step matters

You already completed raw ingestion, profiling, and initial cleaning in the earlier notebook. So now you should continue from the cleaned file, not start from raw data again. The existing project write-up also says cleaned/transformed files are stored under processed-data.

In [2]:
obj = s3.get_object(Bucket=bucket, Key=cleaned_key)
df = pd.read_csv(io.BytesIO(obj["Body"].read()))

print("Loaded cleaned dataset shape:", df.shape)
df.head()

/tmp/ipykernel_278/1276915598.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.BytesIO(obj["Body"].read()))


Loaded cleaned dataset shape: (524878, 12)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,PurchaseYear,PurchaseMonth,PurchaseDay,TotalTransactionValue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-01-12 08:26:00,2.55,17850.0,United Kingdom,2010.0,1.0,12.0,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-01-12 08:26:00,3.39,17850.0,United Kingdom,2010.0,1.0,12.0,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-01-12 08:26:00,2.75,17850.0,United Kingdom,2010.0,1.0,12.0,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-01-12 08:26:00,3.39,17850.0,United Kingdom,2010.0,1.0,12.0,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-01-12 08:26:00,3.39,17850.0,United Kingdom,2010.0,1.0,12.0,20.34


## Step 3: Fix data types and confirm date handling
### What this step does

This converts InvoiceDate into a proper datetime column and checks the schema.

### Why this step matters

Your team already identified invoice date conversion as important for extracting time features and supporting future demand forecasting.

In [3]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")

print(df.dtypes)
print(df["InvoiceDate"].min(), df["InvoiceDate"].max())

Invoice                          object
StockCode                        object
Description                      object
Quantity                          int64
InvoiceDate              datetime64[ns]
Price                           float64
Customer ID                     float64
Country                          object
PurchaseYear                    float64
PurchaseMonth                   float64
PurchaseDay                     float64
TotalTransactionValue           float64
dtype: object
2010-01-12 08:26:00 2011-12-10 17:19:00


## Step 4: Create time-based fields
### What this step does

This creates year, month, week, and quarter features from the invoice date.

### Why this step matters

Your design document says the team wants to use transaction history and engineered time-based features for future demand prediction. These fields are the starting point for that.

In [5]:
# Check how many bad dates exist
print("Missing InvoiceDate values:", df["InvoiceDate"].isna().sum())

# Remove rows where InvoiceDate could not be parsed
df = df.dropna(subset=["InvoiceDate"]).copy()

# Now create date-based fields
iso_calendar = df["InvoiceDate"].dt.isocalendar()

df["PurchaseYear"] = iso_calendar.year.astype(int)
df["PurchaseWeek"] = iso_calendar.week.astype(int)
df["PurchaseMonth"] = df["InvoiceDate"].dt.month.astype(int)
df["Quarter"] = df["InvoiceDate"].dt.quarter.astype(int)

df[["InvoiceDate", "PurchaseYear", "PurchaseWeek", "PurchaseMonth", "Quarter"]].head()

Missing InvoiceDate values: 299454


,InvoiceDate,PurchaseYear,PurchaseWeek,PurchaseMonth,Quarter
0,2010-01-12 08:26:00,2010,2,1,1
1,2010-01-12 08:26:00,2010,2,1,1
2,2010-01-12 08:26:00,2010,2,1,1
3,2010-01-12 08:26:00,2010,2,1,1
4,2010-01-12 08:26:00,2010,2,1,1


## Step 5: Create or confirm the sales amount column
### What this step does

This creates the total value for each transaction line.

### Why this step matters

Your earlier notebook already created TotalTransactionValue. This step makes sure the column exists before aggregation.

In [6]:
if "TotalTransactionValue" not in df.columns:
    df["TotalTransactionValue"] = df["Quantity"] * df["Price"]

df[["Quantity", "Price", "TotalTransactionValue"]].head()

,Quantity,Price,TotalTransactionValue
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34


## Step 6: Keep only the columns needed for forecasting
### What this step does

This selects the fields needed for a weekly product-demand model.

### Why this step matters

For demand forecasting, you do not need everything. The project has already noted that missing Customer ID can limit customer-level analysis, but this assignment is better framed around product demand forecasting and inventory support.

In [7]:
forecast_df = df[[
    "Invoice",
    "StockCode",
    "Quantity",
    "Price",
    "TotalTransactionValue",
    "InvoiceDate",
    "PurchaseYear",
    "PurchaseWeek",
    "PurchaseMonth",
    "Quarter",
    "Country"
]].copy()

forecast_df.head()

,Invoice,StockCode,Quantity,Price,TotalTransactionValue,InvoiceDate,PurchaseYear,PurchaseWeek,PurchaseMonth,Quarter,Country
0,536365,85123A,6,2.55,15.30,2010-01-12 08:26:00,2010,2,1,1,United Kingdom
1,536365,71053,6,3.39,20.34,2010-01-12 08:26:00,2010,2,1,1,United Kingdom
2,536365,84406B,8,2.75,22.00,2010-01-12 08:26:00,2010,2,1,1,United Kingdom
3,536365,84029G,6,3.39,20.34,2010-01-12 08:26:00,2010,2,1,1,United Kingdom
4,536365,84029E,6,3.39,20.34,2010-01-12 08:26:00,2010,2,1,1,United Kingdom


## Step 7: Optional simplification using top products only
### What this step does

This keeps only the top 100 products by quantity sold.

### Why this step matters

Your dataset is large. Keeping only the top products makes the first model much easier to train and explain. This is a very reasonable student-project simplification.

In [8]:
top_n = 100

top_products = (
    forecast_df.groupby("StockCode")["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(top_n)
    .index
)

forecast_df = forecast_df[forecast_df["StockCode"].isin(top_products)].copy()

print("Shape after keeping top products:", forecast_df.shape)
print("Number of products kept:", forecast_df["StockCode"].nunique())

Shape after keeping top products: (34121, 11)
Number of products kept: 100


## Step 8: Aggregate the data to weekly product level
### What this step does

This transforms transaction-level data into a weekly product-demand table.

### Why this step matters

Inventory decisions are based on product demand over time, not on single invoice rows. Also, your design document already points toward inventory optimization and forecast accuracy, which is much better served by weekly demand data.

In [9]:
weekly_df = (
    forecast_df.groupby(["StockCode", "PurchaseYear", "PurchaseWeek"], as_index=False)
    .agg(
        weekly_quantity=("Quantity", "sum"),
        weekly_revenue=("TotalTransactionValue", "sum"),
        avg_price=("Price", "mean"),
        invoice_count=("Invoice", "nunique"),
        purchase_month=("PurchaseMonth", "first"),
        quarter=("Quarter", "first")
    )
)

weekly_df.head()
print("Weekly table shape:", weekly_df.shape)

Weekly table shape: (3472, 9)


## Step 9: Create a sortable weekly date key
### What this step does

This creates a representative date for each year-week combination so you can sort properly and later split chronologically.

### Why this step matters

Forecasting should use time order. So you need a clean time index before making lag features or train/test splits.

In [10]:
weekly_df["week_start_date"] = pd.to_datetime(
    weekly_df["PurchaseYear"].astype(str) + "-W" + weekly_df["PurchaseWeek"].astype(str).str.zfill(2) + "-1",
    format="%G-W%V-%u",
    errors="coerce"
)

weekly_df = weekly_df.sort_values(["StockCode", "week_start_date"]).reset_index(drop=True)

weekly_df[["StockCode", "PurchaseYear", "PurchaseWeek", "week_start_date"]].head(10)

,StockCode,PurchaseYear,PurchaseWeek,week_start_date
0,15034,2010,19,2010-05-10
1,15034,2010,28,2010-07-12
2,15034,2010,32,2010-08-09
3,15034,2010,36,2010-09-06
4,15034,2011,1,2011-01-03
5,15034,2011,2,2011-01-10
6,15034,2011,5,2011-01-31
7,15034,2011,6,2011-02-07
8,15034,2011,10,2011-03-07
9,15034,2011,13,2011-03-28


## Step 10: Create lag features
### What this step does

This creates features showing demand in previous weeks for the same product.

### Why this step matters

This is one of the most useful feature engineering steps in forecasting. Past demand often helps predict future demand.

In [11]:
weekly_df["lag_1_week_qty"] = weekly_df.groupby("StockCode")["weekly_quantity"].shift(1)
weekly_df["lag_2_week_qty"] = weekly_df.groupby("StockCode")["weekly_quantity"].shift(2)
weekly_df["lag_4_week_qty"] = weekly_df.groupby("StockCode")["weekly_quantity"].shift(4)

weekly_df["rolling_4_week_avg_qty"] = (
    weekly_df.groupby("StockCode")["weekly_quantity"]
    .transform(lambda x: x.shift(1).rolling(4).mean())
)

weekly_df.head(10)

,StockCode,PurchaseYear,PurchaseWeek,weekly_quantity,weekly_revenue,avg_price,invoice_count,purchase_month,quarter,week_start_date,lag_1_week_qty,lag_2_week_qty,lag_4_week_qty,rolling_4_week_avg_qty
0,15034,2010,19,2,0.28,0.140000,2,5,2,2010-05-10,NaN,NaN,NaN,NaN
1,15034,2010,28,1,0.85,0.850000,1,7,3,2010-07-12,2.0,NaN,NaN,NaN
2,15034,2010,32,25,4.21,0.495000,2,8,3,2010-08-09,1.0,2.0,NaN,NaN
3,15034,2010,36,1,0.85,0.850000,1,9,3,2010-09-06,25.0,1.0,NaN,NaN
4,15034,2011,1,82,18.38,0.416000,5,1,1,2011-01-03,1.0,25.0,2.0,7.25
5,15034,2011,2,60,8.40,0.140000,1,1,1,2011-01-10,82.0,1.0,1.0,27.25
6,15034,2011,5,240,16.80,0.070000,1,2,1,2011-01-31,60.0,82.0,25.0,42.00
7,15034,2011,6,1236,89.04,0.116667,3,2,1,2011-02-07,240.0,60.0,1.0,95.75
8,15034,2011,10,14,3.34,0.600000,3,3,1,2011-03-07,1236.0,240.0,82.0,404.50
9,15034,2011,13,60,8.40,0.140000,3,4,2,2011-03-28,14.0,1236.0,60.0,387.50


## Step 11: Remove rows that cannot yet use lag features
### What this step does

This drops the first few weeks of each product where lag values are missing.

### Why this step matters

A model cannot train on rows with missing lag features unless you impute them. For a first clean model, dropping those rows is simplest.

In [12]:
model_df = weekly_df.dropna(subset=[
    "lag_1_week_qty",
    "lag_2_week_qty",
    "lag_4_week_qty",
    "rolling_4_week_avg_qty"
]).copy()

print("Model table shape after lag cleanup:", model_df.shape)
model_df.head()

Model table shape after lag cleanup: (3075, 14)


,StockCode,PurchaseYear,PurchaseWeek,weekly_quantity,weekly_revenue,avg_price,invoice_count,purchase_month,quarter,week_start_date,lag_1_week_qty,lag_2_week_qty,lag_4_week_qty,rolling_4_week_avg_qty
4,15034,2011,1,82,18.38,0.416000,5,1,1,2011-01-03,1.0,25.0,2.0,7.25
5,15034,2011,2,60,8.40,0.140000,1,1,1,2011-01-10,82.0,1.0,1.0,27.25
6,15034,2011,5,240,16.80,0.070000,1,2,1,2011-01-31,60.0,82.0,25.0,42.00
7,15034,2011,6,1236,89.04,0.116667,3,2,1,2011-02-07,240.0,60.0,1.0,95.75
8,15034,2011,10,14,3.34,0.600000,3,3,1,2011-03-07,1236.0,240.0,82.0,404.50


## Step 12: One-hot encode the product identifier
### What this step does

This converts StockCode into numeric columns for model training.

### Why this step matters

Machine learning models need numeric input. Since StockCode is categorical, it must be encoded.

In [13]:
model_df = pd.get_dummies(model_df, columns=["StockCode"], drop_first=True)

print("Shape after encoding:", model_df.shape)
model_df.head()

Shape after encoding: (3075, 111)


,PurchaseYear,PurchaseWeek,weekly_quantity,weekly_revenue,avg_price,invoice_count,purchase_month,quarter,week_start_date,lag_1_week_qty,...,StockCode_84945,StockCode_84946,StockCode_84947,StockCode_84978,StockCode_84991,StockCode_84992,StockCode_85099B,StockCode_85099C,StockCode_85099F,StockCode_85123A
4,2011,1,82,18.38,0.416000,5,1,1,2011-01-03,1.0,...,False,False,False,False,False,False,False,False,False,False
5,2011,2,60,8.40,0.140000,1,1,1,2011-01-10,82.0,...,False,False,False,False,False,False,False,False,False,False
6,2011,5,240,16.80,0.070000,1,2,1,2011-01-31,60.0,...,False,False,False,False,False,False,False,False,False,False
7,2011,6,1236,89.04,0.116667,3,2,1,2011-02-07,240.0,...,False,False,False,False,False,False,False,False,False,False
8,2011,10,14,3.34,0.600000,3,3,1,2011-03-07,1236.0,...,False,False,False,False,False,False,False,False,False,False


## Step 13: Define features and target
### What this step does

This separates the predictor columns from the target column.

### Why this step matters

Your target for this project version is weekly quantity sold.

In [14]:
target_col = "weekly_quantity"

drop_cols = [
    "weekly_quantity",
    "week_start_date"
]

X = model_df.drop(columns=drop_cols)
y = model_df[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (3075, 109)
y shape: (3075,)


## Step 14: Split the data chronologically
### What this step does

This splits the dataset into train, validation, and test sets based on time, not randomly.

### Why this step matters

Random splitting leaks future information into training. Since your team wants sales forecast accuracy, the split must reflect real forecasting.

In [15]:
model_df = model_df.sort_values("week_start_date").reset_index(drop=True)

n = len(model_df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = model_df.iloc[:train_end].copy()
val_df   = model_df.iloc[train_end:val_end].copy()
test_df  = model_df.iloc[val_end:].copy()

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (2152, 111)
Validation shape: (461, 111)
Test shape: (462, 111)


## Step 15: Save the split datasets to S3
### What this step does

This writes train, validation, and test CSV files back into your S3 processed-data structure.

### Why this step matters

The assignment explicitly asks you to prepare the data in S3 for training, and your design document already states that this bucket and folder structure is where cleaned/transformed files are stored.

In [16]:
local_train = "/home/sagemaker-user/train.csv"
local_val   = "/home/sagemaker-user/validation.csv"
local_test  = "/home/sagemaker-user/test.csv"

train_df.to_csv(local_train, index=False)
val_df.to_csv(local_val, index=False)
test_df.to_csv(local_test, index=False)

s3.upload_file(local_train, bucket, train_key)
s3.upload_file(local_val, bucket, val_key)
s3.upload_file(local_test, bucket, test_key)

print("Uploaded:")
print(f"s3://{bucket}/{train_key}")
print(f"s3://{bucket}/{val_key}")
print(f"s3://{bucket}/{test_key}")

Uploaded:
s3://retailpulse-team3-ads508/processed-data/train/train.csv
s3://retailpulse-team3-ads508/processed-data/validation/validation.csv
s3://retailpulse-team3-ads508/processed-data/test/test.csv


## Step 16: Build X and y for each split
### What this step does

This prepares train, validation, and test matrices separately.

### Why this step matters

You need these before fitting models and scoring results.

In [17]:
X_train = train_df.drop(columns=["weekly_quantity", "week_start_date"])
y_train = train_df["weekly_quantity"]

X_val = val_df.drop(columns=["weekly_quantity", "week_start_date"])
y_val = val_df["weekly_quantity"]

X_test = test_df.drop(columns=["weekly_quantity", "week_start_date"])
y_test = test_df["weekly_quantity"]

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(2152, 109) (2152,)
(461, 109) (461,)
(462, 109) (462,)


## Step 17: Train a baseline model
### What this step does

This trains a simple linear regression model first.

### Why this step matters

A baseline gives you something to compare against. Then you can show whether a stronger model improved results.

In [18]:
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)

val_pred_baseline = baseline_model.predict(X_val)
test_pred_baseline = baseline_model.predict(X_test)

## Step 18: Evaluate the baseline model
### What this step does

This calculates standard regression metrics.

### Why this step matters

These metrics will help you explain model performance in the notebook and design document.

In [19]:
def regression_metrics(y_true, y_pred, model_name="Model"):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"{model_name} Metrics")
    print("RMSE:", round(rmse, 4))
    print("MAE :", round(mae, 4))
    print("R^2 :", round(r2, 4))
    print("-" * 30)

regression_metrics(y_val, val_pred_baseline, "Baseline Linear Regression on Validation")
regression_metrics(y_test, test_pred_baseline, "Baseline Linear Regression on Test")

Baseline Linear Regression on Validation Metrics
RMSE: 157.8341
MAE : 86.7545
R^2 : 0.5717
------------------------------
Baseline Linear Regression on Test Metrics
RMSE: 175.9634
MAE : 94.3058
R^2 : 0.7629
------------------------------


## Step 19: Train a stronger model
### What this step does

This trains a Random Forest regression model.

### Why this step matters

This is a good next model for structured retail data and is simpler to start with than a full SageMaker managed training job. If your instructor expects notebook-based training inside SageMaker Studio, this still satisfies that.

In [20]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

val_pred_rf = rf_model.predict(X_val)
test_pred_rf = rf_model.predict(X_test)

## Step 20: Evaluate the stronger model
### What this step does

This compares the stronger model against the baseline.

### Why this step matters

Your team can use this comparison to justify which model is better for the business problem.

Code

In [21]:
regression_metrics(y_val, val_pred_rf, "Random Forest on Validation")
regression_metrics(y_test, test_pred_rf, "Random Forest on Test")

Random Forest on Validation Metrics
RMSE: 72.3245
MAE : 27.9791
R^2 : 0.9101
------------------------------
Random Forest on Test Metrics
RMSE: 115.095
MAE : 37.2215
R^2 : 0.8986
------------------------------


## Step 21: Create a comparison table
### What this step does

This makes your notebook easier to present and easier to summarize in the design document.

### Why this step matters

To describe findings from the code. A comparison table makes those findings clear.

In [22]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "Validation_RMSE": [
        np.sqrt(mean_squared_error(y_val, val_pred_baseline)),
        np.sqrt(mean_squared_error(y_val, val_pred_rf))
    ],
    "Validation_MAE": [
        mean_absolute_error(y_val, val_pred_baseline),
        mean_absolute_error(y_val, val_pred_rf)
    ],
    "Validation_R2": [
        r2_score(y_val, val_pred_baseline),
        r2_score(y_val, val_pred_rf)
    ]
})

results

,Model,Validation_RMSE,Validation_MAE,Validation_R2
0,Linear Regression,157.834125,86.754475,0.571721
1,Random Forest,72.324471,27.979089,0.910072


## Step 22: Save model results locally if you want to upload them later
### What this step does

This stores the comparison results as a CSV.

### Why this step matters

It helps with documentation and makes your outputs easier to attach to GitHub or the report.

In [23]:
results_local = "/home/sagemaker-user/model_results.csv"
results.to_csv(results_local, index=False)

print("Saved results to:", results_local)

Saved results to: /home/sagemaker-user/model_results.csv
